In [1]:
# mount google drive
from google.colab import drive
drive.mount('/content/drive')

import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

CODE_DIR = "/content/drive/MyDrive/School Archives/Wentworth Classes/Year 2/Semester 2/Thesis 2/Code Files"
if CODE_DIR not in sys.path:
    sys.path.append(CODE_DIR)

from data_pipeline import get_full_pipeline, get_full_sequence_pipeline
from text_pipeline import load_and_prepare_sequence_data

RESULTS_ROOT = "/content/drive/MyDrive/School Archives/Wentworth Classes/Year 2/Semester 2/Thesis 2/results"
CHART_ROOT = os.path.join(RESULTS_ROOT, "summary_plots")

MODEL_MODE_MAP = {
    "rf": ("network", "base", "unfrozen"),
    "cnnlstm": ("network", "base", "unfrozen"),
    "mae": ("fusion", "base", "unfrozen"),
    "tinyllama": ("fusion", "base", "unfrozen")
}


def get_class_names(model, mode_name):
    plots_dir = os.path.join(RESULTS_ROOT, mode_name, "plots")
    if mode_name == "network" and model == "rf":
        pipe_out = get_full_pipeline(0, batch_size=2048)
        le = pipe_out[-1]
        filename = f"{model}_classes.npy"
        path = os.path.join(plots_dir, filename)
        if not os.path.exists(path):
          np.save(os.path.join(plots_dir, filename), le.classes_)
        return np.load(os.path.join(plots_dir, filename), allow_pickle=True)

    elif mode_name == "network" and model == "cnnlstm":
        filename = f"{model}_classes.npy"
        path = os.path.join(plots_dir, filename)

        if not os.path.exists(path):
          raise FileNotFoundError(f"Class names not found in {path}")

        return np.load(os.path.join(plots_dir, filename), allow_pickle=True)

    elif mode_name == "fusion" and model == "mae":
        pipe_out = get_full_pipeline(2, batch_size=2048)
        le = pipe_out[-1]
        filename = f"{model}_classes.npy"
        path = os.path.join(plots_dir, filename)
        if not os.path.exists(path):
          np.save(os.path.join(plots_dir, filename), le.classes_)
        return np.load(os.path.join(plots_dir, filename), allow_pickle=True)

    elif mode_name == "fusion" and model == "tinyllama":
        tokenizer = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
        _, _, _, _, id2label, _ = load_and_prepare_sequence_data(
            2, tokenizer, max_length=256, window=3
        )
        filename = f"{model}_classes.npy"
        path = os.path.join(plots_dir, filename)
        if not os.path.exists(path):
          np.save(os.path.join(plots_dir, filename), list(id2label.values()))
        return np.load(os.path.join(plots_dir, filename), allow_pickle=True)

    else:
        raise ValueError(f"No class_names logic for model={model}, mode={mode_name}")


def process_cm(model, mode_name, size, encoder):
    plots_dir = os.path.join(RESULTS_ROOT, mode_name, "plots")

    if not os.path.exists(plots_dir):
        print(f"[ERROR] plots_dir missing: {plots_dir}")
        return False

    # build exact filename
    if model in ["mae", "stmae", "distilbert", "tinyllama"]:
        target = f"{model}_{mode_name}_{size}_{encoder}_run_2_cm.npy"
    else:
        target = f"{model}_{mode_name}_{size}_run_2_cm.npy"

    target_path = os.path.join(plots_dir, target)

    if not os.path.exists(target_path):
        print(f"[MISS] {target} not found in {plots_dir}")
        return False

    print(f"[FOUND] {target}")

    save_path = os.path.join(CHART_ROOT, target.replace(".npy", "_top6.png"))

    cm = np.load(target_path)

    class_names = get_class_names(model, mode_name)

    if len(class_names) != cm.shape[0]:
      print(f'[WARN] Class mismatch: {len(class_names)} != {cm.shape[0]}')
      class_names = class_names[:cm.shape[0]]

    print(f"Check {model}-{mode_name}")
    print(f"First 5 classes: {class_names[:5]}")
    print(f"CM shape: {cm.shape}")

    # Normalize
    cm = cm.astype(np.float32)
    cm_norm = cm / cm.sum(axis=1, keepdims=True)
    cm_norm = np.nan_to_num(cm_norm)

    # ---- TOP 6 CLASS FILTERING ----
    # Use original counts (not normalized) to find most frequent classes
    class_counts = cm.sum(axis=1)
    top_k = 6
    top_indices = np.argsort(class_counts)[-top_k:][::-1]

    cm_top = cm_norm[top_indices][:, top_indices]
    class_names_top = [class_names[i] for i in top_indices]

    # ---- CLEAN LABELS ----
    def shorten(name):
        return (
            name.replace("web attack � ", "")
                .replace("dos ", "DoS ")
                .replace("ddos", "DDoS")
                .replace("ftp-patator", "FTP-Patator")
                .replace("ssh-patator", "SSH-Patator")
                .replace(" ", "\n")
        )

    class_names_top = [shorten(c) for c in class_names_top]

    # ---- NICE TITLE FORMATTING ----
    MODEL_NAMES = {
        "rf": "Random Forest",
        "cnnlstm": "CNN-LSTM",
        "mae": "Masked Autoencoder",
        "tinyllama": "TinyLlama"
    }

    MODE_NAMES = {
        "network": "Network Mode",
        "fusion": "Fusion Mode"
    }

    title = f"{MODEL_NAMES.get(model, model)} — {MODE_NAMES.get(mode_name, mode_name)}\n(Top {top_k} Classes, Row-Normalized)"

    # ---- PLOT ----
    plt.figure(figsize=(10, 8))

    sns.heatmap(
        cm_top,
        annot=True,
        fmt=".2f",
        cmap="Blues",
        xticklabels=class_names_top,
        yticklabels=class_names_top,
        vmin=0,
        vmax=1
    )

    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)

    plt.xlabel("Predicted Class")
    plt.ylabel("True Class")
    plt.title(title, fontsize=14)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()

    print(f"[SAVED] {save_path}")
    return True


if not os.path.exists(RESULTS_ROOT):
    raise FileNotFoundError("Results directory not found")

for model, (mode_name, size, encoder) in MODEL_MODE_MAP.items():
    success = process_cm(model, mode_name, size, encoder)

    if not success:
        print(f"[SKIP] {model} ({mode_name})")

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[FOUND] rf_network_base_run_2_cm.npy
Building dataset from raw files...
target
benign      1384122
dos hulk     120992
ddos         103610
normal        35000
backdoor      14000
Name: count, dtype: int64
target
benign        296598
dos hulk       25927
ddos           22202
normal          7500
ransomware      3000
Name: count, dtype: int64
target
benign      296598
dos hulk     25927
ddos         22202
normal        7500
dos           3000
Name: count, dtype: int64
['backdoor', 'benign', 'bot', 'ddos', 'dos', 'dos goldeneye', 'dos hulk', 'dos slowhttptest', 'dos slowloris', 'ftp-patator', 'infiltration', 'injection', 'mitm', 'normal', 'password', 'portscan', 'ransomware', 'scanning', 'ssh-patator', 'web attack � brute force', 'web attack � sql injection', 'web attack � xss', 'xss']
['backdoor', 'benign', 'bot', 'ddos', 'dos', 'dos g

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Loading cache data from /content/drive/MyDrive/School Archives/Wentworth Classes/Year 2/Semester 2/Thesis 2/cache/text_raw_mode_2_ms5.parquet...
Loading splits from /content/drive/MyDrive/School Archives/Wentworth Classes/Year 2/Semester 2/Thesis 2/cache/text_splits_sequence_mode_2_w3...
Loading label map from /content/drive/MyDrive/School Archives/Wentworth Classes/Year 2/Semester 2/Thesis 2/cache/label_sequence_map_mode_2.json...
Loading tokenized data from /content/local_cache/tokenized_sequence_mode_2_TinyLlama-TinyLlama-1.1B-intermediate-step-1431k-3T_max256_w3...
[WARN] Class mismatch: 25 != 11
Check tinyllama-fusion
First 5 classes: ['Benign' 'Bot' 'DDoS' 'DoS GoldenEye' 'DoS Hulk']
CM shape: (11, 11)
[SAVED] /content/drive/MyDrive/School Archives/Wentworth Classes/Year 2/Semester 2/Thesis 2/results/summary_plots/tinyllama_fusion_base_unfrozen_run_2_cm_top6.png
